In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
%%capture
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [3]:
pip install "datasets<4.0.0"   # latest 3.x still supports loading scripts

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import torch
from datasets import load_dataset, concatenate_datasets
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer, SFTConfig
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


In [5]:
print("Authenticating with Hugging Face...")
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

Authenticating with Hugging Face...


In [8]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
HF_HUB_ID = "kanishkav/qwen2.5-1.5B-tuned" 
N_SAMPLES = 60_000
MAX_SEQ_LENGTH = 2048
tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",
)

In [1]:
print("Loading Unsloth optimized model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    fast_inference = True,
    load_in_4bit=False,  
)

Loading Unsloth optimized model...


NameError: name 'FastLanguageModel' is not defined

In [9]:
print("Applying Unsloth LoRA...")
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,      
    bias="none",
    use_gradient_checkpointing="unsloth", 
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

Applying Unsloth LoRA...


Unsloth 2026.6.7 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [10]:
print("Loading FLAN CoT subsets (chiayewken/flan-cot)...")

configs = ['aqua', 'creak', 'ecqa', 'esnli', 'gsm8k', 'qasc', 'qed', 'sensemaking', 'strategyqa']
dataset_list = []

for config in configs:
    ds = load_dataset("chiayewken/flan-cot", config, split="train")
    dataset_list.append(ds)

raw_dataset = concatenate_datasets(dataset_list)

print(f"Total samples available across all configs: {len(raw_dataset)}")
print(f"Shuffling and selecting exactly {N_SAMPLES} samples...")
dataset = raw_dataset.shuffle(seed=42).select(range(N_SAMPLES))

Loading FLAN CoT subsets (chiayewken/flan-cot)...


README.md: 0.00B [00:00, ?B/s]

aqua/train-00000-of-00001.parquet:   0%|          | 0.00/440k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2728 [00:00<?, ? examples/s]

creak/train-00000-of-00001.parquet:   0%|          | 0.00/824k [00:00<?, ?B/s]

...

sensemaking/train-00000-of-00001.parquet:   0%|          | 0.00/588k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6070 [00:00<?, ? examples/s]

strategyqa/train-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2061 [00:00<?, ? examples/s]

Total samples available across all configs: 74771
Shuffling and selecting exactly 60000 samples...


In [11]:
def format_chat_template(example):
    # Safely extract the prompt and response to prevent KeyErrors across different configs
    prompt = example.get("inputs", example.get("input", example.get("source", "")))
    response = example.get("targets", example.get("target", example.get("rationale", "")))
    
    messages = [
        {"role": "system", "content": "You are a helpful reasoning assistant."},
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": response}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

print("Applying chat template mapping...")
dataset = dataset.map(format_chat_template, num_proc=4, desc="Formatting chat template")

Applying chat template mapping...


Formatting chat template (num_proc=4):   0%|          | 0/60000 [00:00<?, ? examples/s]

In [12]:
sft_cfg = SFTConfig(
    output_dir="qwen1.5b-unsloth-sft",
    num_train_epochs=2,                 
    per_device_train_batch_size=4,      
    gradient_accumulation_steps=8,      
    learning_rate=2e-4,                 
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    max_grad_norm=1.0,
    max_seq_length=MAX_SEQ_LENGTH,                
    packing=True,                       
    fp16=True, 
    optim="adamw_torch",                
    logging_steps=10,
    save_strategy="epoch",
    seed=42,
    dataset_text_field="text",
    
    # --- Hugging Face Hub Specific Arguments ---
    push_to_hub=True,
    hub_model_id=HF_HUB_ID,
    hub_strategy="checkpoint",
    hub_token=hf_token,
)

# --- 8. Initialize Trainer & Train ---
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    tokenizer=tokenizer,
    args=sft_cfg
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/60000 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=8):   0%|          | 0/60000 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


In [13]:
print("Starting fast training with Unsloth on merged FLAN CoT (FP16)...")
trainer.train()

print(f"Pushing final adapters to Hub: {HF_HUB_ID}...")
model.push_to_hub(HF_HUB_ID, token=hf_token)
tokenizer.push_to_hub(HF_HUB_ID, token=hf_token)

print("Training complete and successfully pushed to Hugging Face Hub.")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting fast training with Unsloth on merged FLAN CoT (FP16)...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 664 | Num Epochs = 2 | Total steps = 42
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,2.998481
20,0.001672
30,0.000385
40,0.000291


...

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.
Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpwwsr42cs/tokenizer_config.json.


Saved model to https://huggingface.co/kanishkav/qwen2.5-1.5B-tuned


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Training complete and successfully pushed to Hugging Face Hub.
